# Movie Recommendation System

This notebook explains how to build a movie recommendation system using `movies.csv` and `ratings.csv`.

## 1. Load the dataset

In [ ]:
import pandas as pd

movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

print(movies.shape)
print(ratings.shape)

movies.head(), ratings.head()

## 2. Dataset overview

In [ ]:
print("Number of movies:", movies["movieId"].nunique())
print("Number of users:", ratings["userId"].nunique())
print("Number of ratings:", len(ratings))
print("Average rating:", round(ratings["rating"].mean(), 2))

ratings["rating"].describe()

## 3. Popularity-based recommendation

In [ ]:
stats = (
    ratings.groupby("movieId")
    .agg(avg_rating=("rating", "mean"), rating_count=("rating", "count"))
    .reset_index()
)

C = stats["avg_rating"].mean()
m = stats["rating_count"].quantile(0.75)

stats["weighted_score"] = (
    (stats["rating_count"] / (stats["rating_count"] + m)) * stats["avg_rating"]
    + (m / (stats["rating_count"] + m)) * C
)

popular_movies = (
    stats.merge(movies, on="movieId")
    .sort_values("weighted_score", ascending=False)
)

popular_movies[["title", "genres", "avg_rating", "rating_count", "weighted_score"]].head(10)

## 4. Content-based filtering using genres

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)

tfidf = TfidfVectorizer(stop_words="english")
content_matrix = tfidf.fit_transform(movies["genres_clean"])

content_knn = NearestNeighbors(metric="cosine", algorithm="brute")
content_knn.fit(content_matrix)

def recommend_by_content(title, n=10):
    matches = movies[movies["title"].str.lower().str.contains(title.lower(), na=False)]
    if matches.empty:
        return "Movie not found"

    movie_index = matches.index[0]
    distances, indices = content_knn.kneighbors(content_matrix[movie_index], n_neighbors=n+1)

    result = []
    for distance, index in zip(distances.flatten(), indices.flatten()):
        if index != movie_index:
            result.append({
                "title": movies.iloc[index]["title"],
                "genres": movies.iloc[index]["genres"],
                "similarity": round(1 - distance, 4)
            })

    return pd.DataFrame(result)

recommend_by_content("Toy Story", n=10)

## 5. Collaborative filtering using user ratings

In [ ]:
from scipy.sparse import csr_matrix

user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
).fillna(0)

movie_ids = list(user_movie_matrix.columns)
movie_id_to_index = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

item_user_matrix = csr_matrix(user_movie_matrix.T.values)

cf_knn = NearestNeighbors(metric="cosine", algorithm="brute")
cf_knn.fit(item_user_matrix)

def recommend_for_user(user_id, n=10, min_rating=4.0):
    user_ratings = ratings[ratings["userId"] == user_id]
    seen_movies = set(user_ratings["movieId"].unique())
    liked_movies = user_ratings[user_ratings["rating"] >= min_rating]

    scores = {}

    for _, row in liked_movies.iterrows():
        movie_id = row["movieId"]
        rating = row["rating"]

        if movie_id not in movie_id_to_index:
            continue

        index = movie_id_to_index[movie_id]
        distances, indices = cf_knn.kneighbors(item_user_matrix[index], n_neighbors=30)

        for distance, idx in zip(distances.flatten(), indices.flatten()):
            candidate_movie_id = movie_ids[idx]

            if candidate_movie_id in seen_movies:
                continue

            similarity = 1 - distance
            scores[candidate_movie_id] = scores.get(candidate_movie_id, 0) + similarity * rating

    recommendations = (
        pd.DataFrame([{"movieId": mid, "score": score} for mid, score in scores.items()])
        .merge(movies, on="movieId")
        .sort_values("score", ascending=False)
        .head(n)
    )

    return recommendations[["title", "genres", "score"]]

recommend_for_user(user_id=1, n=10)

## 6. Conclusion

This project includes three recommendation strategies: popularity-based, content-based, and collaborative filtering. The Streamlit app in `app.py` allows users to interact with the system.